In [8]:
### Load the environment variables

from dotenv import load_dotenv

load_dotenv()

True

#### ***1.Document Ingestion***

#### ***Load the all files and convert that data into the LangChain Document Object.***

In [1]:
###path Exists
import os
path = "../kubernetes"

if os.path.exists(path):
    print("Path Is Exist.")
else:
    print("Path is not Exist.")

Path Is Exist.


In [3]:
### Load all pdf files using DirectoryLoader by PyMuPdfLoader

from langchain_community.document_loaders import PyMuPDFLoader,DirectoryLoader

loader = DirectoryLoader(
    path,
    glob = "*.pdf",
    loader_cls=PyMuPDFLoader
)

documents = loader.load()


print("Number Of Documents:",len(documents))

Number Of Documents: 3983


#### ***2.Document Splitting***

##### **Document Splitting is a process of split documents into chunks based on chunk_size and chunk overlap using Recursive Character Text Splitter, it will preseve the structure and meaning of the data like paragraphs, lines,sentences, and more.**

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = splitter.split_documents(documents)

print("Number Of Chunks:",len(chunks))

Number Of Chunks: 9507


In [5]:
### Lets test the single chunk randomly

chunks[9]

Document(metadata={'producer': 'WeasyPrint 56.1', 'creator': '', 'creationdate': '', 'source': '..\\kubernetes\\Concepts.pdf', 'file_path': '..\\kubernetes\\Concepts.pdf', 'total_pages': 676, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 3}, page_content='Kubernetes provides you with:\nService discovery and load balancing Kubernetes can expose a container using the\nDNS name or using their own IP address. If traffic to a container is high, Kubernetes is\nable to load balance and distribute the network traffic so that the deployment is stable.\nStorage orchestration Kubernetes allows you to automatically mount a storage system\nof your choice, such as local storages, public cloud providers, and more.\nAutomated rollouts and rollbacks You can describe the desired state for your deployed\ncontainers using Kubernetes, and it can change the actual state to the desired state at a\ncontro

In [6]:
### page content
chunks[9].page_content

'Kubernetes provides you with:\nService discovery and load balancing Kubernetes can expose a container using the\nDNS name or using their own IP address. If traffic to a container is high, Kubernetes is\nable to load balance and distribute the network traffic so that the deployment is stable.\nStorage orchestration Kubernetes allows you to automatically mount a storage system\nof your choice, such as local storages, public cloud providers, and more.\nAutomated rollouts and rollbacks You can describe the desired state for your deployed\ncontainers using Kubernetes, and it can change the actual state to the desired state at a\ncontrolled rate. For example, you can automate Kubernetes to create new containers for\nyour deployment, remove existing containers and adopt all their resources to the new\ncontainer.\nAutomatic bin packing You provide Kubernetes with a cluster of nodes that it can use\nto run containerized tasks. You tell Kubernetes how much CPU and memory (RAM) each'

In [7]:
### metadata
chunks[9].metadata

{'producer': 'WeasyPrint 56.1',
 'creator': '',
 'creationdate': '',
 'source': '..\\kubernetes\\Concepts.pdf',
 'file_path': '..\\kubernetes\\Concepts.pdf',
 'total_pages': 676,
 'format': 'PDF 1.7',
 'title': '',
 'author': '',
 'subject': '',
 'keywords': '',
 'moddate': '',
 'trapped': '',
 'modDate': '',
 'creationDate': '',
 'page': 3}

#### ***3.Embeddings.***
##### **Embedding is a numerical representation of actual text.**

In [9]:
###create a embedding by using langchain hugging face.

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
        model = "BAAI/bge-large-en-v1.5",
        model_kwargs = {"device":"cpu"},
        encode_kwargs = {"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [11]:
### Using batching technique to create embedding vector for chunks
from langchain_chroma import Chroma
batch_size = 200
batch_chunk = chunks[:200]
vectorstore = Chroma.from_documents(
    documents = batch_chunk,
    embedding=embedding_model,
    persist_directory="../vectorstore/kubernetes_rag",
    collection_name="kubernetes_rag"
)

In [12]:

for i in range(batch_size,len(chunks),batch_size):

    batch_chunk = chunks[i:i+batch_size]

    vectorstore.add_documents(batch_chunk)

    min_dx = min(i+batch_size,len(chunks))

    print(f"Added vectors succesfully from {i} to {min_dx-1}")
    
print("Added all vectors succesfully")

Added vectors succesfully from 200 to 399
Added vectors succesfully from 400 to 599
Added vectors succesfully from 600 to 799
Added vectors succesfully from 800 to 999
Added vectors succesfully from 1000 to 1199
Added vectors succesfully from 1200 to 1399
Added vectors succesfully from 1400 to 1599
Added vectors succesfully from 1600 to 1799
Added vectors succesfully from 1800 to 1999
Added vectors succesfully from 2000 to 2199
Added vectors succesfully from 2200 to 2399
Added vectors succesfully from 2400 to 2599
Added vectors succesfully from 2600 to 2799
Added vectors succesfully from 2800 to 2999
Added vectors succesfully from 3000 to 3199
Added vectors succesfully from 3200 to 3399
Added vectors succesfully from 3400 to 3599
Added vectors succesfully from 3600 to 3799
Added vectors succesfully from 3800 to 3999
Added vectors succesfully from 4000 to 4199
Added vectors succesfully from 4200 to 4399
Added vectors succesfully from 4400 to 4599
Added vectors succesfully from 4600 to 4

In [13]:
vectorstore._collection.count()

9507

In [14]:
question = "What is a Kubernetes Deployment?"

In [15]:
docs = vectorstore.similarity_search(question,k=5)
docs

[Document(id='e4a8d10a-aea4-4e18-8e23-feff97928e35', metadata={'author': '', 'moddate': '', 'keywords': '', 'creationdate': '', 'creator': '', 'title': '', 'page': 1, 'source': '..\\kubernetes\\Concepts.pdf', 'file_path': '..\\kubernetes\\Concepts.pdf', 'modDate': '', 'trapped': '', 'format': 'PDF 1.7', 'creationDate': '', 'producer': 'WeasyPrint 56.1', 'subject': '', 'total_pages': 676}, page_content='Kubernetes is a portable, extensible, open source platform for managing containerized\nworkloads and services, that facilitates both declarative configuration and automation. It has a\nlarge, rapidly growing ecosystem. Kubernetes services, support, and tools are widely available.\nThe name Kubernetes originates from Greek, meaning helmsman or pilot. K8s as an\nabbreviation results from counting the eight letters between the "K" and the "s". Google open-\nsourced the Kubernetes project in 2014. Kubernetes combines over 15 years of Google\'s\nexperience running production workloads at scal